# Tackle Noise with Error Correction
**Computer Laboratory II (417526) – Quantum AI – Practical 2**

Install once (Linux terminal):
```
pip install qiskit qiskit-aer matplotlib pylatexenc
```
Runs on the local Aer simulator – no IBM account or API token needed (the lab manual's `execute` / `IBMQ` / `qiskit.aqua` / `qiskit.ignis` code was removed from current Qiskit, so this notebook uses the current API).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__)

def run(qc, shots=1024, backend=sim, **kw):
    # Transpile for the backend, run, and return the counts dictionary.
    t = transpile(qc, backend, **kw)
    return backend.run(t, shots=shots).result().get_counts()

## Practical 2 – Tackle Noise with Error Correction
**Theory:** Quantum error correction (QEC) encodes one *logical* qubit into several *physical* qubits so that errors can be detected and corrected. The simplest code is the **3-qubit bit-flip (repetition) code**:

* Encode: |ψ⟩ → copy the value onto 3 qubits with two CNOTs (|1⟩ → |111⟩)
* Noise: each physical qubit may suffer a bit flip (X error) with probability *p*
* Decode: two CNOTs compute the syndrome, a Toffoli (CCX) does the majority-vote correction

If only one qubit flips, the code corrects it. The logical error probability becomes ≈ 3p² − 2p³, which is lower than *p* whenever *p* < 0.5.

**Part A** simulates this with an Aer noise model. **Part B** shows *measurement-error mitigation*, which replaces the manual's removed `qiskit.ignis` code.

In [ ]:
from qiskit_aer.noise import NoiseModel, pauli_error, ReadoutError

def bitflip_noise_model(p):
    # Bit-flip error with probability p applied on every 'id' (identity) gate
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(pauli_error([('X', p), ('I', 1 - p)]), ['id'])
    return nm

def unprotected_error(p, shots=4000):
    qc = QuantumCircuit(1, 1)
    qc.x(0)                 # logical |1>
    qc.id(0)                # noisy channel
    qc.measure(0, 0)
    noisy = AerSimulator(noise_model=bitflip_noise_model(p))
    counts = run(qc, shots, noisy, optimization_level=0)
    return 1 - counts.get('1', 0) / shots

def protected_circuit():
    qc = QuantumCircuit(3, 1)
    qc.x(0)                              # logical |1>
    qc.cx(0, 1); qc.cx(0, 2)             # ENCODE  |1> -> |111>
    qc.barrier()
    for i in range(3):
        qc.id(i)                         # NOISY CHANNEL (bit flip on each qubit)
    qc.barrier()
    qc.cx(0, 1); qc.cx(0, 2)             # SYNDROME
    qc.ccx(1, 2, 0)                      # CORRECT (majority vote)
    qc.measure(0, 0)
    return qc

def protected_error(p, shots=4000):
    noisy = AerSimulator(noise_model=bitflip_noise_model(p))
    counts = run(protected_circuit(), shots, noisy, optimization_level=0)
    return 1 - counts.get('1', 0) / shots

display(protected_circuit().draw(output='mpl'))

In [ ]:
ps = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]
unprot = [unprotected_error(p) for p in ps]
prot   = [protected_error(p) for p in ps]
theory = [3*p**2 - 2*p**3 for p in ps]

print(f"{'p':>5} | {'no QEC':>8} | {'with QEC':>9} | {'theory 3p^2-2p^3':>16}")
for p, u, e, t in zip(ps, unprot, prot, theory):
    print(f"{p:5.2f} | {u:8.4f} | {e:9.4f} | {t:16.4f}")

plt.figure(figsize=(6, 4))
plt.plot(ps, unprot, 'o-', label='Without error correction')
plt.plot(ps, prot,   's-', label='With 3-qubit code')
plt.plot(ps, theory, '--', color='gray', label='Theory 3p²−2p³')
plt.xlabel('Physical error probability p'); plt.ylabel('Logical error probability')
plt.legend(); plt.grid(alpha=.3); plt.title('Bit-flip code vs unprotected qubit')
plt.show()

### Part B – Measurement-error mitigation (replacement for `qiskit.ignis`)
Readout noise makes a qubit that is really |0⟩ get read as 1 (and vice versa). We:
1. Prepare each of the 8 basis states and measure → build a **calibration matrix** `A` where `A[i, j] = P(read i | prepared j)`
2. Measure the real circuit (GHZ state `000 + 111`) → noisy distribution `m`
3. Solve `A · x = m` for the corrected distribution `x`

In [ ]:
n = 3
ro = ReadoutError([[0.95, 0.05],     # P(read 0|0), P(read 1|0)
                   [0.10, 0.90]])    # P(read 0|1), P(read 1|1)
nm = NoiseModel(); nm.add_all_qubit_readout_error(ro)
noisy = AerSimulator(noise_model=nm)
shots = 20000

# Circuit of the manual: GHZ state -> ideal outcome is only 000 and 111
qc = QuantumCircuit(n, n)
qc.h(0); qc.cx(0, 1); qc.cx(0, 2)
qc.measure(range(n), range(n))

ideal    = run(qc, shots, sim)
raw      = run(qc, shots, noisy)

# 1) calibration matrix
labels = [format(i, f'0{n}b') for i in range(2**n)]
A = np.zeros((2**n, 2**n))
for j, lab in enumerate(labels):
    cal = QuantumCircuit(n, n)
    for k, bit in enumerate(reversed(lab)):      # qubit k <-> k-th bit from the right
        if bit == '1':
            cal.x(k)
    cal.measure(range(n), range(n))
    for out, v in run(cal, shots, noisy).items():
        A[int(out, 2), j] = v / shots

# 2) + 3) mitigate
m = np.array([raw.get(l, 0) / shots for l in labels])
x = np.linalg.solve(A, m)
x = np.clip(x, 0, None); x /= x.sum()
mitigated = {l: float(v) for l, v in zip(labels, x)}

to_prob = lambda d: {k: v / sum(d.values()) for k, v in d.items()}
print("Calibration matrix:\n", np.round(A, 3))
print("\nIdeal     :", to_prob(ideal))
print("Original  :", to_prob(raw))
print("Mitigated :", {k: round(v, 3) for k, v in mitigated.items() if v > 0.001})
plot_histogram([to_prob(raw), mitigated], legend=['Original (noisy)', 'Mitigated'])